# galform_execution — Examples

`galform_execution` automates the generation and submission of SLURM batch scripts for GALFORM N-body runs on COSMA. It generates self-contained `tcsh` scripts from Python-controlled configuration, replacing the legacy `qsub_galform_Nbody_example.csh` workflow.

This notebook walks through the full API from config browsing to job submission.

**Contents**
1. [Setup](#1-setup)
2. [Browsing available configurations](#2-browsing-available-configurations)
3. [Dry run — preview before submitting](#3-dry-run)
4. [Inspecting the generated SLURM script](#4-inspecting-the-generated-slurm-script)
5. [Choosing a simulation](#5-choosing-a-simulation)
6. [Choosing a model](#6-choosing-a-model)
7. [Customising pipeline stages with RunFlags](#7-customising-pipeline-stages-with-runflags)
8. [Overriding GALFORM input parameters](#8-overriding-galform-input-parameters)
9. [Multi-output redshifts](#9-multi-output-redshifts)
10. [Galaxy and halo trees](#10-galaxy-and-halo-trees)
11. [Custom output paths and SLURM settings](#11-custom-output-paths-and-slurm-settings)
12. [Retry configuration](#12-retry-configuration)
13. [Submitting for real](#13-submitting-for-real)
14. [CLI quick reference](#14-cli-quick-reference)

---
## 1. Setup

In [ ]:
from galform_execution.submit_galform_job import (
    GalformSubmitter,
    RunFlags,
    SIMULATION_CONFIGS,
    MODEL_CONFIGS,
    DUST_CONFIGS,
    _default_galform_dir,
)

# Path to your compiled GALFORM source tree on COSMA.
# Defaults to /cosma/apps/durham/$USER/galform — override if needed.
galform_dir = _default_galform_dir()

---
## 2. Browsing available configurations

Simulation, model, and dust profiles are all driven by JSON files under
`galform_execution/config/`. You can inspect them at runtime without
constructing a submitter.

In [ ]:
# All built-in simulations
print(f"{'Simulation':<25} {'Snapshots (iz_list)':<45} {'Subvolumes'}")
print("-" * 80)
for name, cfg in sorted(SIMULATION_CONFIGS.items()):
    iz_str = str(cfg.iz_list) if cfg.iz_list else "(not set)"
    if len(iz_str) > 42:
        iz_str = iz_str[:39] + "..."
    print(f"{name:<25} {iz_str:<45} {cfg.nvol_range}")

In [ ]:
# All built-in models
print(f"{'Model':<30} {'Base input file':<50} {'Dust profile'}")
print("-" * 95)
for name, cfg in sorted(MODEL_CONFIGS.items()):
    dust_label = f"fcloud={cfg.dust_params.fcloud}"
    print(f"{name:<30} {cfg.base_inputs_file:<50} {dust_label}")

In [ ]:
# Dust profiles
print(f"{'Profile':<15} {'fcloud':<10} {'rfacburst':<12} {'tesc_disk':<12} {'tesc_burst'}")
print("-" * 60)
for name, dust in DUST_CONFIGS.items():
    print(f"{name:<15} {dust.fcloud:<10} {dust.rfacburst:<12} {dust.tesc_disk:<12} {dust.tesc_burst}")

---
## 3. Dry run

Pass `dry_run=True` to `submit_all_jobs` to print the generated SLURM script
to stdout without submitting anything. This is the safest way to check your
configuration before touching the queue.

In [ ]:
submitter = GalformSubmitter(
    galform_dir,
    nbody_sim="Mill2",
    model="lc16",
    iz=40,
    nvol="1-64",
    output_folder_name="MyProject",
)

# Prints the full tcsh script; returns [] without submitting.
submitter.submit_all_jobs(dry_run=True)

---
## 4. Inspecting the generated SLURM script

`create_slurm_script(iz)` returns the script as a string so you can inspect,
diff, or archive it programmatically.

In [ ]:
script = submitter.create_slurm_script(iz=40)
print(script)

In [ ]:
# Save to disk for review or archiving
with open("Mill2_lc16_iz40.tcsh", "w") as fh:
    fh.write(script)
print("Saved Mill2_lc16_iz40.tcsh")

---
## 5. Choosing a simulation

Any key from `SIMULATION_CONFIGS` works as `nbody_sim`. Each simulation
knows its own default `iz_list` and `nvol_range`, so you often don't need
to specify them explicitly.

In [ ]:
# P-Millennium (L800) — 1024 subvolumes, EAGLE cosmology
submitter_l800 = GalformSubmitter(
    galform_dir,
    nbody_sim="L800",
    model="lc16.newmg",
    output_folder_name="L800_Run",
)
print(f"L800: iz_list={submitter_l800.iz_list}, nvol_range={submitter_l800.nvol_range}")

In [ ]:
# Original Millennium (Mill1)
submitter_mill1 = GalformSubmitter(
    galform_dir,
    nbody_sim="Mill1",
    model="lc16",
    output_folder_name="Mill1_Run",
)
print(f"Mill1: iz_list={submitter_mill1.iz_list}, nvol_range={submitter_mill1.nvol_range}")

In [ ]:
# EAGLE dark-matter-only
submitter_eagle = GalformSubmitter(
    galform_dir,
    nbody_sim="EagleDM",
    model="lc16",
    output_folder_name="EAGLE_Run",
)
print(f"EagleDM: iz_list={submitter_eagle.iz_list}, nvol_range={submitter_eagle.nvol_range}")

In [ ]:
# COLIBRE L100
submitter_colibre = GalformSubmitter(
    galform_dir,
    nbody_sim="COLIBRE-L100m6",
    model="lc16",
    output_folder_name="COLIBRE_Run",
)
print(f"COLIBRE-L100m6: iz_list={submitter_colibre.iz_list}, nvol_range={submitter_colibre.nvol_range}")

In [ ]:
# FLAMINGO L1000
submitter_flamingo = GalformSubmitter(
    galform_dir,
    nbody_sim="FLAMINGO-L1000N1800",
    model="lc16",
    output_folder_name="FLAMINGO_Run",
)
print(f"FLAMINGO: iz_list={submitter_flamingo.iz_list}, nvol_range={submitter_flamingo.nvol_range}")

In [ ]:
# Override the default snapshot list for any simulation
submitter_custom_iz = GalformSubmitter(
    galform_dir,
    nbody_sim="L800",
    model="lc16.newmg",
    iz_list=[271, 207, 176],   # Only submit these three snapshots
    output_folder_name="L800_HighZ",
)
print(f"Custom iz_list: {submitter_custom_iz.iz_list}")

---
## 6. Choosing a model

Any key from `MODEL_CONFIGS` works as `model`. Each model carries a base
`.input.ref` file, a dust profile, and optional extra parameter replacements.

In [ ]:
# Bower et al. (2006) on MilliMillennium
submitter_b06 = GalformSubmitter(
    galform_dir,
    nbody_sim="MilliMil",
    model="b06",
    output_folder_name="Bower06_Run",
)

In [ ]:
# González-Perez et al. (2014) on MillGas
submitter_gp14 = GalformSubmitter(
    galform_dir,
    nbody_sim="MillGas",
    model="gp14",
    output_folder_name="GP14_Run",
)

In [ ]:
# Lacey et al. (2016) with updated merger history on L800
submitter_lc16_newmg = GalformSubmitter(
    galform_dir,
    nbody_sim="L800",
    model="lc16.newmg",
    output_folder_name="LC16_newmg_Run",
)

In [ ]:
# GP14 with saturated feedback (extra_replacements baked into model)
submitter_satf = GalformSubmitter(
    galform_dir,
    nbody_sim="MillGas",
    model="gp14.satf",
    output_folder_name="GP14_SatF_Run",
)
# gp14.satf already sets Saturate_Feedback=.true. and thresholdVcirc=65.0
# via extra_replacements in models.json — no manual overrides needed.

---
## 7. Customising pipeline stages with RunFlags

`RunFlags` controls which post-processing executables run after GALFORM.
The defaults come from `config/run_flags.json`; you can override any subset
by constructing a `RunFlags` object and passing it to the submitter.

In [ ]:
# Inspect the current defaults
from galform_execution.submit_galform_job import load_run_flags_config
defaults = load_run_flags_config()
print(defaults)

In [ ]:
# Enable dust properties and z=0 galaxy sample; disable luminosity function
flags = RunFlags(
    galform=True,
    neta=True,
    lum_fun=False,
    study_stellar_mass_function=True,
    dust_props=True,   # run dust properties calculation
    samp_z0=True,      # produce z=0 galaxy catalogue
)

submitter_flags = GalformSubmitter(
    galform_dir,
    nbody_sim="Mill2",
    model="lc16",
    iz=67,
    nvol="1-64",
    run_flags=flags,
    output_folder_name="Mill2_DustProps",
)
submitter_flags.submit_all_jobs(dry_run=True)

In [ ]:
# Run GALFORM only — skip all post-processing (useful for debugging)
flags_galform_only = RunFlags(
    galform=True,
    neta=False,
    lum_fun=False,
    study_stellar_mass_function=False,
)

submitter_galform_only = GalformSubmitter(
    galform_dir,
    nbody_sim="Mill2",
    model="lc16",
    iz=40,
    nvol="1-4",
    run_flags=flags_galform_only,
    output_folder_name="Mill2_Debug",
)
submitter_galform_only.submit_all_jobs(dry_run=True)

---
## 8. Overriding GALFORM input parameters

`input_overrides` injects arbitrary `name → value` pairs into the GALFORM
parameter file via `replace_variable.csh`, after the model's base replacements.
Use this for parameter studies or one-off tweaks without creating a new model.

In [ ]:
# Parameter study: vary supernova feedback velocities
submitter_vhot = GalformSubmitter(
    galform_dir,
    nbody_sim="Mill2",
    model="lc16",
    iz=40,
    nvol="1-64",
    input_overrides={
        "vhotdisk": "290",
        "vhotburst": "290",
    },
    output_folder_name="Mill2_vhot290",
)
submitter_vhot.submit_all_jobs(dry_run=True)

In [ ]:
# Tweak AGN parameters
submitter_agn = GalformSubmitter(
    galform_dir,
    nbody_sim="L800",
    model="lc16.newmg",
    input_overrides={
        "f_q": "10.0",
        "alpha_cool": "0.8",
    },
    output_folder_name="L800_AGN_study",
)

In [ ]:
# Switch to a single IMF
submitter_nmf = GalformSubmitter(
    galform_dir,
    nbody_sim="Mill2",
    model="lc16",
    input_overrides={"nmf": "1"},
    output_folder_name="Mill2_singleIMF",
)

---
## 9. Multi-output redshifts

By default each job outputs at a single redshift (`nout=1`, `zout=$z`).
You can request multiple output redshifts per job in two ways:

- **`output_iz_list`** — list of snapshot indices; the corresponding redshifts
  are read from the simulation's snapshot file automatically.
- **`output_redshifts`** — list of explicit redshift values.

Both set `nout` and populate `zout` in the GALFORM parameter file.

In [ ]:
# Output at four snapshots in one GALFORM run
submitter_multi_iz = GalformSubmitter(
    galform_dir,
    nbody_sim="Mill2",
    model="lc16",
    iz=67,               # Single SLURM job, uses z from iz=67
    nvol="1-64",
    output_iz_list=[20, 30, 40, 67],   # nout=4; zout resolved from Mill2.txt
    output_folder_name="Mill2_MultiOut",
)
submitter_multi_iz.submit_all_jobs(dry_run=True)

In [ ]:
# Or specify redshifts directly
submitter_multi_z = GalformSubmitter(
    galform_dir,
    nbody_sim="Mill2",
    model="lc16",
    iz=67,
    nvol="1-64",
    output_redshifts=[0.0, 0.5, 1.0, 2.0, 3.0],
    output_folder_name="Mill2_MultiZ",
)

---
## 10. Galaxy and halo trees

To build galaxy merger trees across multiple output snapshots, set
`build_galaxy_trees = .true.` via `input_overrides`.

When `build_galaxy_trees` is enabled **and** more than one output snapshot
is requested, `GalformSubmitter` automatically injects
`mgalmin_output_descendants = .true.` so descendants are tracked across
outputs — you do not need to set this manually.

In [ ]:
submitter_trees = GalformSubmitter(
    galform_dir,
    nbody_sim="Mill2",
    model="lc16",
    iz=67,
    nvol="1-64",
    output_iz_list=[20, 40, 67],
    input_overrides={
        "build_galaxy_trees": ".true.",
        "output_halo_trees": ".true.",
    },
    output_folder_name="Mill2_Trees",
)

# mgalmin_output_descendants is injected automatically — verify:
print(submitter_trees.input_overrides)

In [ ]:
submitter_trees.submit_all_jobs(dry_run=True)

---
## 11. Custom output paths and SLURM settings

All path and SLURM parameters can be overridden. Defaults target COSMA5
(`/cosma5/data/durham/$USER`); adjust for COSMA7/8 or custom allocations.

In [ ]:
# COSMA7 with a dp004 project allocation
submitter_c7 = GalformSubmitter(
    galform_dir,
    nbody_sim="L800",
    model="lc16.newmg",
    output_base_dir="/cosma7/data/dp004/dc-hick2",
    output_folder_name="Production_v2",
    log_path="/cosma7/data/dp004/dc-hick2/logs/Production_v2",
    partition="cosma7",
    account="dp004",
    walltime="48:00:00",
)
print(f"Output dir: {submitter_c7.models_dir}")
print(f"Log path:   {submitter_c7.log_path}")

In [ ]:
# COSMA8
submitter_c8 = GalformSubmitter(
    galform_dir,
    nbody_sim="FLAMINGO-L1000N1800",
    model="lc16",
    output_base_dir="/cosma8/data/dp004/dc-hick2",
    output_folder_name="FLAMINGO_Production",
    partition="cosma8",
    account="dp004",
    walltime="72:00:00",
)

In [ ]:
# Use a custom GALFORM executable (e.g. a development build)
submitter_dev = GalformSubmitter(
    galform_dir,
    nbody_sim="Mill2",
    model="lc16",
    iz=40,
    nvol="1-4",
    galform_exe="/cosma/apps/durham/dc-hick2/galform_dev/build/galform2",
    output_folder_name="Mill2_DevBuild",
)

---
## 12. Retry configuration

SLURM can transiently reject submissions under scheduler pressure.
`GalformSubmitter` retries with exponential backoff when it detects
these transient errors. The defaults (4 retries, 15 s base delay, ×2
backoff) are conservative; increase them for large array submissions.

In [ ]:
# More aggressive retry for large production runs
submitter_retry = GalformSubmitter(
    galform_dir,
    nbody_sim="L800",
    model="lc16.newmg",
    output_folder_name="L800_Production",
    submit_retries=8,
    submit_retry_delay_s=30.0,
    submit_retry_backoff=2.0,
)
print(f"Retries: {submitter_retry.submit_retries}, base delay: {submitter_retry.submit_retry_delay_s}s")

---
## 13. Submitting for real

Once you are satisfied with the dry run output, drop `dry_run=True` (or pass
`dry_run=False`) to actually submit to SLURM. `submit_all_jobs` returns the
list of job IDs returned by `sbatch`.

In [ ]:
submitter = GalformSubmitter(
    galform_dir,
    nbody_sim="Mill1",
    model="lc16",
    iz=63,
    nvol="1-64",
    output_folder_name="Galform_Test_Notebook",
)

job_ids = submitter.submit_all_jobs(dry_run=False)
print(f"Submitted {len(job_ids)} job array(s): {job_ids}")

In [ ]:
# Submit a single snapshot instead of all iz_list entries
job_id = submitter.submit_job(iz=63, dry_run=False)
print(f"Job ID: {job_id}")

---
## 14. CLI quick reference

The `submit-galform-job` command exposes the same functionality from the
terminal. All examples below are illustrative — adjust paths and parameters
to match your environment.

### Discover available simulations and models
```bash
submit-galform-job --list-simulations
submit-galform-job --list-models
```

### Dry run — preview the generated script
```bash
submit-galform-job /path/to/galform \
    --nbody-sim Mill2 --model lc16 \
    --iz 40 --nvol 1-64 \
    --dry-run
```

### Basic submission
```bash
submit-galform-job /path/to/galform \
    --nbody-sim Mill2 --model lc16 \
    --iz 40 --nvol 1-64 \
    --output-folder-name MyProject
```

### Custom SLURM partition and walltime
```bash
submit-galform-job /path/to/galform \
    --nbody-sim L800 --model lc16.newmg \
    --output-folder-name Production_v2 \
    --output-base-dir /cosma7/data/dp004/dc-hick2 \
    --partition cosma7 --account dp004 \
    --walltime 48:00:00
```

### Override snapshot list
```bash
submit-galform-job /path/to/galform \
    --nbody-sim L800 --model lc16.newmg \
    --iz-list 271 207 176 155
```

### Multi-output redshifts
```bash
# By snapshot index (redshifts resolved from snapshot file)
submit-galform-job /path/to/galform \
    --nbody-sim Mill2 --model lc16 \
    --iz 67 --output-iz-list 20 40 67

# By explicit redshift values
submit-galform-job /path/to/galform \
    --nbody-sim Mill2 --model lc16 \
    --iz 67 --output-z-list 0.0 0.5 1.0 2.0
```

### Pipeline stage toggles
```bash
# Disable neta and luminosity function; enable dust properties and z=0 sample
submit-galform-job /path/to/galform \
    --nbody-sim Mill2 --model lc16 \
    --no-neta --no-lum-fun \
    --run-dust-props --run-samp-z0
```

### Galaxy trees
```bash
submit-galform-job /path/to/galform \
    --nbody-sim Mill2 --model lc16 \
    --iz 67 --output-iz-list 20 40 67 \
    --build-galaxy-trees --output-halo-trees
```

### Custom GALFORM executable
```bash
submit-galform-job /path/to/galform \
    --nbody-sim Mill2 --model lc16 \
    --galform-exe /path/to/dev/build/galform2 \
    --iz 40 --nvol 1-4 --dry-run
```